# Notebook to analyse sourcephotonly_w_miri fits!

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import pickle as pkl
import pandas as pd
import os

import scipy
import numpy as np
import astropy
import scipy.stats as stats

from astropy.table import Table
from prospector_utils.analysis import analyse_miri_fits
from prospector_utils.plotting import *

quiescent = [7549, 8013, 8469, 9395, 10128, 10339, 10400, 10565, 10592, 11142, 11494, 16419, 18668, 21477]
no_spec = [9517, 9809, 11051, 11451, 12133, 17713, 17984, 20195, 20693, 20720, 21472, 22990]
#below_ms = [10600, 18977, 21451]

print(f"I should exclude {len(no_spec)} galaxies from my analysis with Prospector.")

In [ ]:
table_path = '/Users/benjamincollins/University/master/Red_Cardinal/photometry/phot_tables/fits/Phot_Table_MIRI.fits'

table = Table.read(table_path, format='fits')
galaxy_ids = np.asarray([str(gid) for gid in table['ID']])

plot_dir = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/appphot_only_wMIRI/fits/'
appphot_only_wMIRI = '/Users/benjamincollins/Data/Bluejay/Prospector/v2.0.5/appphot_only_wMIRI_cat/'
stats_dir = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/appphot_only_wMIRI/pickle_files/'
phot_table = '/Users/benjamincollins/University/Master/Red_Cardinal/photometry/phot_tables/fits/Phot_Table_MIRI.fits'

In [ ]:
analyse_miri_fits(phot_table=phot_table,
                  data_dir=appphot_only_wMIRI,
                  plot_dir=plot_dir,
                  stats_dir=stats_dir,
                  add_dust=True)

# Suppressing dust emission in 17517

In [ ]:
import prospect.io.read_results as reader
from prospector_utils.params import get_MAP
from prospect.sources import FastStepBasis
from prospect.utils.plotting import posterior_samples

plot_dir = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/appphot_only_wMIRI/fits_nodust/'
stats_dir = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/appphot_only_wMIRI/pickle_files_nodust/'
phot_table = '/Users/benjamincollins/University/Master/Red_Cardinal/photometry/phot_tables/fits/Phot_Table_MIRI.fits'

# Load the h5 file for the given objid
data_dir = '/Users/benjamincollins/Data/Bluejay/Prospector/v2.0.5/appphot_only_wMIRI_cat/'

objid = 17517

h5_file = glob.glob(os.path.join(data_dir, f"output_{objid}*.h5"))

try:
    h5_file = h5_file[0]
except IndexError:
    print(f"No PROSPECTOR results found for objid {objid}.")

# Load PROSPECTOR results
full_path = os.path.join(data_dir, h5_file)
results, obs, model = reader.results_from(full_path)

filename = os.path.join(stats_dir, f"{objid}.pkl")

#print(obs['filters'])
#return

# Now we have to exclude the last 3 parameters from the fit
map_parameters = get_MAP(results)

#print(map_parameters)

# Build the MAP dictionary
MAP = {}
for a,b in zip(results['theta_labels'], map_parameters):
    MAP[a] = b
    
zred = MAP['zred']
logmass = MAP['logmass']
dust2 = MAP['dust2']    # extract the diffuse dust V-band optical depth

# Calculate the spectrum based on the Maximum A Posteriori (MAP) parameters
sps = FastStepBasis(zcontinuous=1)

# 1. Create a copy of your MAP parameters
no_dust_params = map_parameters.copy()

# 2. Toggle the model setting to False
# This prevents the code from adding the IR 'glow'
model.params['add_dust_emission'] = np.array([False])

# 3. Predict the spectrum
# The resulting spec/phot will show the attenuated UV but NO IR emission
spec, phot, _ = model.predict(no_dust_params, obs=obs, sps=sps)    

# Convert maggies to µJy
maggies_to_muJy = 3631e6

# Wavelengths of the model spectrum
wave_spec = sps.wavelengths

# Convert to arrays
phot = np.array(phot)

# Draw 100 posterior samples
samples = posterior_samples(results, 100)

sample_specs = []
for params_i in samples:    
    spec_i, _, _ = model.predict(params_i, obs=obs, sps=sps)
    sample_specs.append(spec_i)
sample_specs = np.array(sample_specs)  # shape: (nsample, nwave)

# Takes the per-pixel percentiles such that the final spectra are not actual spectra of Prospectors parameter space
lower = np.percentile(sample_specs, 16, axis=0)
median = np.percentile(sample_specs, 50, axis=0)
upper = np.percentile(sample_specs, 84, axis=0)

# Compute filter wavelength in microns
phot_wave = np.array([filt.wave_effective for filt in obs['filters']])  # in Angstroms

data = {
    # Important metadata
    'id': objid,
    'zred': zred,
    'maggies_to_muJy': maggies_to_muJy,
    
    'model': {
        'spec_best': spec,
        'spec_16th': lower,
        'spec_median': median,
        'spec_84th': upper,
        'wave_spec': wave_spec,
        'sample_specs': sample_specs[:10],
        'phot': phot,
        'phot_wave': phot_wave
    },
    
    # One entry for the observation dictionary
    'obs': obs,
    
    #'fit_quality': fit_quality,
    
    'galaxy_properties': {
        'logmass': logmass,
        'dust2': dust2
    },
    
    # Moved it outside so it's easier accessible
    'map_theta': MAP # Keep the full raw dictionary just in case    
}

# Write output to a pickle file
with open(filename, 'wb') as f:
    pkl.dump(data, f)
print(f"💾 Saved data to {filename}")
    
if plot_dir:
    plot_miri_fit(filename, plot_dir)


# Compare MIRI parameters with no MIRI

In [ ]:
filename = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/sourcephotonly/pickle_files/17517.pkl'

with open(filename, 'rb') as f:
    fit_data = pkl.load(f)

print(fit_data)

Restructure pickle files to have map_parameters easily accessible:

In [ ]:
stats_dir = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/sourcephotonly/pickle_files'

output_folder = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/sourcephotonly/pickle_files_new'

os.makedirs(output_folder, exist_ok=True)

file_paths = glob.glob(os.path.join(stats_dir, "*.pkl"))
print(f"Found {len(file_paths)} files. Starting restructuring...")

for f_path in file_paths:
    with open(f_path, 'rb') as f:
        old_data = pkl.load(f)

    # 1. Extract raw MAP dictionary
    gal_props = old_data.get('galaxy_properties', {})
    
    MAP = gal_props.get('map_theta', {})
    
    # 2. Extract observational data
    # If obs_miri isn't a separate key yet, we take it from 'obs'
    obs_data = old_data.get('obs', {})

    # 3. Reconstruct Galaxy Properties
    # We pull these from the existing map_theta
    galaxy_props = {
        'logmass': gal_props.get('logmass'),
        'dust2': gal_props.get('dust2'),
        'sfr_100myr': gal_props.get('sfr_100myr'), # Call helper for math
        'sfr_30myr': gal_props.get('srf_30myr'), 
        'sfr_bins': gal_props.get('sfr_bins'), # Add your specific bin array if available
        'agebins': gal_props.get('agebins')   # Add your specific agebins array if available
    }

    # 4. Create New Structure
    new_data = {
        'id': old_data.get('id'),
        'zred': old_data.get('zred'),
        'maggies_to_muJy': old_data.get('maggies_to_muJy'),
        'model': old_data.get('model'),
        'obs': obs_data,
        'fit_quality': old_data.get('fit_quality', {}), # Carry over if exists
        'galaxy_properties': galaxy_props,
        'map_theta': MAP # Now outside and easily accessible
    }

    # 5. Save the new file
    new_filename = os.path.join(output_folder, os.path.basename(f_path))
    with open(new_filename, 'wb') as f:
        pkl.dump(new_data, f)

print("Restructuring complete.")